In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
import time
from tqdm import tqdm
import os
from dotenv import load_dotenv



GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

# Common endpoint and headers
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
HEADERS = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

# Output Normalization

def clean_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip().lower()

    # Handle short answers a/b/c
    if text in ["a", "a.", "option a"]:
        return "a. City name available and main city matching"
    elif text in ["b", "b.", "option b"]:
        return "b. City name available but main city not matching"
    elif text in ["c", "c.", "option c"]:
        return "c. No City name in description"

    # Handle longer answers
    if "main city matching" in text:
        return "a. City name available and main city matching"
    elif "not matching" in text:
        return "b. City name available but main city not matching"
    elif "no city" in text:
        return "c. No City name in description"

    # Fallback: return raw text
    return text
# ================================
# Function to interact with Llama3 (Model 1)
def chat_with_llm_model1(user_input):
    response = requests.post(
        GROQ_URL,
        headers=HEADERS,
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a city-checking assistant.\n"
                        "Return ONLY one of the following outputs:\n"
                        "a. City name available and main city matching\n"
                        "b. City name available but main city not matching\n"
                        "c. No City name in description\n"
                        "Do not return anything else."
                    )
                },
                {"role": "user", "content": user_input}
            ],
            "max_tokens": 50,
            "temperature": 0.0
        },
    )

    if response.status_code == 200:
        raw = response.json()["choices"][0]["message"]["content"]
        return clean_output(raw)
    else:
        return f"Error: {response.status_code} - {response.text}"


# Function to interact with Mixtral (Model 2)
# Model 2 → Llama 3.3

def chat_with_llm_model2(user_input):
    response = requests.post(
        GROQ_URL,
        headers=HEADERS,
        json={
            "model": "llama-3.3-70b-versatile",
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a city-checking assistant.\n"
                        "Return ONLY one of the following outputs:\n"
                        "a. City name available and main city matching\n"
                        "b. City name available but main city not matching\n"
                        "c. No City name in description\n"
                        "Do not return anything else."
                    )
                },
                {"role": "user", "content": user_input}
            ],
            "max_tokens": 50,
            "temperature": 0.0
        },
    )

    if response.status_code == 200:
        raw = response.json()["choices"][0]["message"]["content"]
        return clean_output(raw)
    else:
        return f"Error: {response.status_code} - {response.text}"

    
# Load the Excel file and the specific sheet
file_path = r"C:\Users\Sarwesh sasi\Downloads\city_name.xlsx"
sheet_name = 'Cosmo Input Report'
print("Loading Excel file.")
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Load the prompt from the text file
print("Loading prompt from text file.")
with open(r"C:\Users\Sarwesh sasi\Downloads\prompt.txt", 'r') as file:
    prompt_template = file.read()

# Check for result columns and add them if they don't exist
for col in ['Result_Model1', 'Result_Model2', 'City_Check']:
    if col not in df.columns:
        df[col] = None
        
MAX_THREADS = 5
matches = 0
# Function to process each row and update results for both models=
def process_row(index, row):
    main_city = row['Source City name']
    description = row['Description']
    prompt = prompt_template.format(main_city=main_city, description=description)

    result_model1 = chat_with_llm_model1(prompt)
    result_model2 = chat_with_llm_model2(prompt)
    time.sleep(2)
    
    city_name_lower = main_city.lower()
    description_lower = description.lower()
    city_check = (city_name_lower in description_lower) or (f"{city_name_lower}'s" in description_lower)
    
    city_check_result = "City name available and matching" if city_check else "City name not available or mismatched"
    
    return index, result_model1, result_model2, city_check_result

# Parallel processing of the rows
total_rows = len(df)
print("Starting parallel processing of rows.")
with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = {executor.submit(process_row, i, row): i for i, row in df.iterrows()}
    for future in tqdm(futures, total=total_rows, desc="Processing Rows"):
        i, result_model1, result_model2, city_check_result = future.result()
        df.at[i, 'Result_Model1'] = result_model1
        df.at[i, 'Result_Model2'] = result_model2
        df.at[i, 'City_Check'] = city_check_result
    print("finished processing all rows")

def verify_model_results(row):
    expected_result = "a. City name available and main city matching"
    city_check_phrase = "City name available and matching"
    model1_status = "Correct" if row['Result_Model1'] == expected_result else "Result_Model1 is wrong"
    model2_status = "Correct" if row['Result_Model2'] == expected_result else "Result_Model2 is wrong"
    overall_status = ""

    if row['City_Check'] == city_check_phrase:
        if row['Result_Model1'] != expected_result:
            overall_status += model1_status
        if row['Result_Model2'] != expected_result:
            overall_status += " and " + model2_status if overall_status else model2_status
    else:
        # If City_Check is not as expected, no comparison is required with model results.
        overall_status = "City not matching, no need to check models."
    
    return overall_status if overall_status else "Correct"

# Ensure the fourth column exists
if 'Result_Check' not in df.columns:
    df['Result_Check'] = None

print("Verifying model results and updating 'Result_Check' column.")
df['Result_Check'] = df.apply(verify_model_results, axis=1)

# Calculate summary information
correct_count = df['Result_Check'].value_counts().get("Correct", 0)
wrong_model1_count = df['Result_Check'].str.contains("Result_Model1 is wrong").sum()
wrong_model2_count = df['Result_Check'].str.contains("Result_Model2 is wrong").sum()
no_check_needed_count = df['Result_Check'].str.contains("City not matching").sum()

# Output summary information
print(f"Total rows processed: {len(df)}")
print(f"Correct matches: {correct_count}")
print(f"Rows where Result_Model1 is wrong: {wrong_model1_count}")
print(f"Rows where Result_Model2 is wrong: {wrong_model2_count}")
print(f"Rows where model comparison was not needed: {no_check_needed_count}")

# Save the modified DataFrame back to Excel
print("Saving the results to Excel.")
df.to_excel(r"D:\Data_Analyst\LLM Aari.xlsx", sheet_name=sheet_name, index=False)
print("Processing completed and file saved.")

Loading Excel file.
Loading prompt from text file.
Starting parallel processing of rows.


Processing Rows: 100%|██████████| 48/48 [00:23<00:00,  2.06it/s]

finished processing all rows
Verifying model results and updating 'Result_Check' column.
Total rows processed: 48
Correct matches: 0
Rows where Result_Model1 is wrong: 40
Rows where Result_Model2 is wrong: 40
Rows where model comparison was not needed: 8
Saving the results to Excel.
Processing completed and file saved.


In [4]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
import time
from tqdm import tqdm
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Data_Analyst\LLM Project\.env", override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY was not loaded from the .env file.")

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
HEADERS = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

def clean_output(text: str) -> str:
    if not text:
        return ""
    text = text.strip().lower()

    if text in ["a", "a.", "option a"] or "main city matching" in text:
        return "a. City name available and main city matching"
    elif text in ["b", "b.", "option b"] or "not matching" in text:
        return "b. City name available but main city not matching"
    elif text in ["c", "c.", "option c"] or "no city" in text:
        return "c. No City name in description"
    return text

def chat_with_llm(model_name, user_input):
    response = requests.post(
        GROQ_URL, headers=HEADERS,
        json={
            "model": model_name,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a city-checking assistant.\n"
                        "Return ONLY one of the following outputs:\n"
                        "a. City name available and main city matching\n"
                        "b. City name available but main city not matching\n"
                        "c. No City name in description\n"
                        "Do not return anything else."
                    )
                },
                {"role": "user", "content": user_input}
            ],
            "max_tokens": 500,
            "temperature": 0.0
        },
    )
    if response.status_code == 200:
        raw_content = response.json()["choices"][0]["message"]["content"]
        return clean_output(raw_content)
    else:
        return f"Error: {response.status_code} - {response.text}"

file_path = r"C:\Users\Sarwesh sasi\Downloads\city_name.xlsx"
sheet_name = 'Cosmo Input Report'
print("Loading Excel file.")
df = pd.read_excel(file_path, sheet_name=sheet_name)

print("Loading prompt from text file.")
with open(r"C:\Users\Sarwesh sasi\Downloads\prompt.txt", 'r') as file:
    prompt_template = file.read()

for col in ['Result_Model1', 'Result_Model2', 'City_Check']:
    if col not in df.columns:
        df[col] = None
        
MAX_THREADS = 1  
def process_row(index, row):
    main_city = row['Source City name']
    description = row['Description']
    prompt = prompt_template.format(main_city=main_city, description=description)

    # Comparing Qwen 27B against GPT-OSS 20B
    result_model1 = chat_with_llm("qwen/qwen3.8-27b", prompt)
    result_model2 = chat_with_llm("openai/gpt-oss-20b", prompt)
    
    # Delay to remain within Groq API rate limits
    time.sleep(4.5) 
    
    city_name_lower = str(main_city).lower()
    description_lower = str(description).lower()
    city_check = (city_name_lower in description_lower) or (f"{city_name_lower}'s" in description_lower)
    city_check_result = "City name available and matching" if city_check else "City name not available or mismatched"
    
    return index, result_model1, result_model2, city_check_result

total_rows = len(df)
print("Starting processing of rows (Slowed down to bypass Rate Limits).")
with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = {executor.submit(process_row, i, row): i for i, row in df.iterrows()}
    for future in tqdm(futures, total=total_rows, desc="Processing Rows"):
        i, result_model1, result_model2, city_check_result = future.result()
             
        df.at[i, 'Result_Model1'] = result_model1
        df.at[i, 'Result_Model2'] = result_model2
        df.at[i, 'City_Check'] = city_check_result
    print("finished processing all rows")

def verify_model_results(row):
    expected_result = "a. City name available and main city matching"
    city_check_phrase = "City name available and matching"
    model1_status = "Correct" if row['Result_Model1'] == expected_result else "Result_Model1 is wrong"
    model2_status = "Correct" if row['Result_Model2'] == expected_result else "Result_Model2 is wrong"
    overall_status = ""

    if row['City_Check'] == city_check_phrase:
        if row['Result_Model1'] != expected_result:
            overall_status += model1_status
        if row['Result_Model2'] != expected_result:
            overall_status += " and " + model2_status if overall_status else model2_status
    else:
        overall_status = "City not matching, no need to check models."
    
    return overall_status if overall_status else "Correct"

if 'Result_Check' not in df.columns:
    df['Result_Check'] = None

print("Verifying model results and updating 'Result_Check' column.")
df['Result_Check'] = df.apply(verify_model_results, axis=1)

correct_count = df['Result_Check'].value_counts().get("Correct", 0)
wrong_model1_count = df['Result_Check'].str.contains("Result_Model1 is wrong").sum()
wrong_model2_count = df['Result_Check'].str.contains("Result_Model2 is wrong").sum()
no_check_needed_count = df['Result_Check'].str.contains("City not matching").sum()

print(f"Total rows processed: {len(df)}")
print(f"Correct matches: {correct_count}")
print(f"Rows where Result_Model1 is wrong: {wrong_model1_count}")
print(f"Rows where Result_Model2 is wrong: {wrong_model2_count}")
print(f"Rows where model comparison was not needed: {no_check_needed_count}")

print("Saving the results to Excel.")
df.to_excel(r"D:\Data_Analyst\LLM Project\LLM Final Output.xlsx", sheet_name=sheet_name, index=False)
print("Processing completed and file saved.")

Loading Excel file.
Loading prompt from text file.
Starting processing of rows (Slowed down to bypass Rate Limits).


Processing Rows: 100%|██████████| 48/48 [04:52<00:00,  6.09s/it]

finished processing all rows
Verifying model results and updating 'Result_Check' column.
Total rows processed: 48
Correct matches: 40
Rows where Result_Model1 is wrong: 0
Rows where Result_Model2 is wrong: 0
Rows where model comparison was not needed: 8
Saving the results to Excel.
Processing completed and file saved.


In [9]:
import requests

url = "https://api.groq.com/openai/v1/models"

response = requests.get(
    url,
    headers=HEADERS,
    timeout=30
)

print("HTTP Status:", response.status_code)

if response.status_code == 200:

    available_models = {
        model["id"]
        for model in response.json().get("data", [])
    }

    for model_id in [
        "openai/gpt-oss-120b",
        "qwen/qwen3.8-27b"
    ]:

        status = (
            "AVAILABLE"
            if model_id in available_models
            else "NOT LISTED"
        )

        print(f"{model_id}: {status}")

else:
    print("Error response:", response.text)

HTTP Status: 200
openai/gpt-oss-120b: AVAILABLE
qwen/qwen3.8-27b: AVAILABLE


In [5]:
test_response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers=HEADERS,
    timeout=30
)

print("Authentication status:", test_response.status_code)

Authentication status: 200


In [1]:
import os
import requests
from dotenv import load_dotenv

# 1. Setup Authentication
load_dotenv(r"D:\Data_Analyst\LLM Project\.env", override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    print("❌ ERROR: API Key not found. Check your .env file.")
else:
    HEADERS = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

    # 2. Check which models your API key ACTUALLY has access to right now:
    print("Fetching authorized models for your API key...")
    models_response = requests.get("https://api.groq.com/openai/v1/models", headers=HEADERS)

    if models_response.status_code == 200:
        available_models = [m['id'] for m in models_response.json().get('data', [])]
        print("✅ You have access to these models:")
        for m in available_models:
            print(f" - {m}")
        
        # 3. Make 1 test request using the first model in your list
        if available_models:
            # You can manually change this to any model name printed above
            test_model_name = available_models[0] 
            print(f"\n--- Testing 1 single output using: {test_model_name} ---")
            
            sample_input = "1. main city = Albuquerque\n2. description = Generic Copy: Albuquerque's early morning news at 4:30AM."
            
            response = requests.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers=HEADERS,
                json={
                    "model": test_model_name,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "You are a city-checking assistant.\n"
                                "Return ONLY one of the following outputs:\n"
                                "a. City name available and main city matching\n"
                                "b. City name available but main city not matching\n"
                                "c. No City name in description\n"
                                "Do not return anything else."
                            )
                        },
                        {"role": "user", "content": sample_input}
                    ],
                    "max_tokens": 50,
                    "temperature": 0.0
                }
            )
            
            if response.status_code == 200:
                print("\n✅ SUCCESS! The model replied:")
                print(response.json()["choices"][0]["message"]["content"])
            else:
                print(f"\n❌ ERROR {response.status_code}: {response.text}")

    else:
        print(f"❌ Failed to fetch models. Error {models_response.status_code}: {models_response.text}")

Fetching authorized models for your API key...
✅ You have access to these models:
 - openai/gpt-oss-20b
 - allam-2-7b
 - qwen/qwen3.8-27b
 - openai/gpt-oss-120b
 - meta-llama/llama-prompt-guard-2-86m
 - whisper-large-v3
 - whisper-large-v3-turbo
 - openai/gpt-oss-safeguard-20b
 - meta-llama/llama-prompt-guard-2-22m
 - canopylabs/orpheus-v1-english
 - canopylabs/orpheus-arabic-saudi

--- Testing 1 single output using: openai/gpt-oss-20b ---

✅ SUCCESS! The model replied:

